# Experiment 3-B1 — Fresh 30-Seed Validation of the Frozen B=1 Comparison

This notebook is an **independent fresh-routing-seed validation** of the already-selected
B=1 comparison from `experiment3_targeted_hotspot_repair.ipynb`.
It is **not** a new repair search.

## Frozen comparison (selected BEFORE any fresh evaluation)
For each of the 24 B=1 starting cases, exactly three fixed alternatives are evaluated:
1. **Original** — the original minimum-width GidNET reuse plan (Experiment 2 `reuse_chains`).
2. **Repair** — the previously selected TOP structural / targeted generalized repair
   (`experiment3_generalized_candidates.csv`, `selected == True`, `repair_rank == 1`).
3. **Restart-1** — the previously selected single GidNET restart baseline
   (`experiment3_restart_generation.csv`, `candidate_index == 1`, deterministic
   regeneration with the stored `first_restart_seed`).

## Absolute constraints (enforced by construction)
- No new repair candidates, no re-ranking, no new restarts, no selection changes.
- Fresh seeds `100..129` are used **only** for evaluation, never for selection.
- `experiment3_targeted_hotspot_repair.ipynb` is never modified (only read).
- No file under `experiment3_results/` is overwritten. All new outputs go to
  `experiment3_b1_fresh30_validation/` only.
- Old routing seeds `42-46` are never mixed into fresh means.

## Execution order
Run cells top-to-bottom. The expensive 30-seed routing cell is gated behind the
Step-2 verification cell: it refuses to run unless all 72 fixed alternatives
(24 Original + 24 Repair + 24 Restart-1) reconstruct exactly.


In [1]:
from pathlib import Path
import ast
import copy
import json
import logging
import math
import random
import time

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# ---- Routing seeds: old (frozen evaluation) vs fresh (this validation) ----
OLD_ROUTING_SEEDS = [42, 43, 44, 45, 46]
FRESH_ROUTING_SEEDS = list(range(100, 130))  # exactly 100..129 inclusive
LOGICAL_DEPTH_TOLERANCE = 0.05  # same guard as the original Experiment 3

assert OLD_ROUTING_SEEDS == [42, 43, 44, 45, 46]
assert FRESH_ROUTING_SEEDS == list(range(100, 130)) and len(FRESH_ROUTING_SEEDS) == 30

ROOT = Path.cwd()
OUT_DIR = ROOT / "experiment3_b1_fresh30_validation"
FIG_DIR = OUT_DIR / "figures"
# Only the NEW results directory is ever created/written.
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PARTIAL_PATH = OUT_DIR / "experiment3_b1_fresh30_partial_checkpoint.csv"

def find_file(filename):
    matches = list(ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} anywhere under: {ROOT}")
    if len(matches) > 1:
        print(f"[WARNING] Multiple copies of {filename} found (using first):")
        for p in matches:
            print("   ", p)
    return matches[0]

EXP1_RAW_PATH = find_file("experiment1_raw.csv")
EXP1_PLAN_PATH = find_file("experiment1_plan_summary.csv")
EXP2_PLAN_PATH = find_file("experiment2_plan_features.csv")
EXP3_GEN_CAND_PATH = ROOT / "experiment3_results" / "experiment3_generalized_candidates.csv"
EXP3_RESTART_GEN_PATH = ROOT / "experiment3_results" / "experiment3_restart_generation.csv"
EXP3_BUDGET_RAW_PATH = ROOT / "experiment3_results" / "experiment3_budget_raw_routing.csv"
EXP3_BUDGET_SUMMARY_PATH = ROOT / "experiment3_results" / "experiment3_budget_summary.csv"

import qiskit
print("python + qiskit provenance: qiskit", qiskit.__version__)
print("Old routing seeds:", OLD_ROUTING_SEEDS)
print("Fresh routing seeds: 100..129 (%d seeds)" % len(FRESH_ROUTING_SEEDS))
print("New output directory:", OUT_DIR)


python + qiskit provenance: qiskit 0.45.3
Old routing seeds: [42, 43, 44, 45, 46]
Fresh routing seeds: 100..129 (30 seeds)
New output directory: C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment3_b1_fresh30_validation


## Step 0 — Inspect the original Experiment 3 (source of truth)

`experiment3_targeted_hotspot_repair.ipynb` and its saved CSVs determine the frozen
selection. This cell only **reads** those files and prints an inventory:
which files/columns define the B=1 repair selection, the restart baseline, and the old routing.


In [2]:
# ------------------------------------------------------------
# STEP 0 — inventory of the original Experiment 3 outputs (READ ONLY)
# ------------------------------------------------------------
exp1_raw = pd.read_csv(EXP1_RAW_PATH)
exp1_plans = pd.read_csv(EXP1_PLAN_PATH)
exp2_plans = pd.read_csv(EXP2_PLAN_PATH)
gen_candidates = pd.read_csv(EXP3_GEN_CAND_PATH)
restart_gen = pd.read_csv(EXP3_RESTART_GEN_PATH)
budget_raw = pd.read_csv(EXP3_BUDGET_RAW_PATH)
budget_summary = pd.read_csv(EXP3_BUDGET_SUMMARY_PATH)

print("=== Saved-file inventory (all pre-existing, none modified) ===")
for name, df in [
    ("experiment1_raw.csv", exp1_raw),
    ("experiment1_plan_summary.csv", exp1_plans),
    ("experiment2_plan_features.csv", exp2_plans),
    ("experiment3_generalized_candidates.csv", gen_candidates),
    ("experiment3_restart_generation.csv", restart_gen),
    ("experiment3_budget_raw_routing.csv", budget_raw),
    ("experiment3_budget_summary.csv", budget_summary),
]:
    print(f"{name}: shape={df.shape}")

print("\n--- key column sets ---")
print("exp1_raw:", exp1_raw.columns.tolist())
print("exp2_plans (first 12):", exp2_plans.columns.tolist()[:12], "...")
print("gen_candidates:", gen_candidates.columns.tolist())
print("restart_gen:", restart_gen.columns.tolist())
print("budget_raw:", budget_raw.columns.tolist())

# ---- B=1 repair selection: structurally minimal, pre-routing ----
selected_repairs = gen_candidates[gen_candidates["selected"] == True].copy()
print("\n=== B=1 repair selection ===")
print("selected repairs:", len(selected_repairs))
print("repair_rank values among selected:", sorted(selected_repairs["repair_rank"].unique().tolist()))
assert len(selected_repairs) == 24, "Expected 24 selected B=1 repairs."
assert (selected_repairs["repair_rank"] == 1.0).all(), "Every B=1 repair must be structural rank 1."

# ---- B=1 restart baseline: candidate_index == 1 ----
restart_b1 = restart_gen[restart_gen["candidate_index"] == 1].copy()
print("\n=== B=1 restart baseline (candidate_index == 1) ===")
print("restart rows:", len(restart_b1), "| successes:", int(restart_b1['success'].sum()))
print("same_plan_as_start count:", int(restart_b1["same_plan_as_start"].fillna(False).sum()))
assert len(restart_b1) == 24 and bool(restart_b1["success"].all())

# ---- repair/restart seeds must cover the SAME 24 starting cases ----
repair_keys = set(zip(selected_repairs["circuit"], selected_repairs["seed_plan_id"]))
restart_keys = set(zip(restart_b1["circuit"], restart_b1["seed_plan_id"]))
print("\nrepair starting cases:", len(repair_keys), "| restart starting cases:", len(restart_keys))
assert repair_keys == restart_keys, "Repair and restart must share the same 24 starting cases."

# ---- old 5-seed routing coverage for the B=1 slice ----
old_repair_b1 = budget_raw[(budget_raw["strategy"] == "repair") & (budget_raw["candidate_index"] == 1)]
old_restart_b1 = budget_raw[(budget_raw["strategy"] == "restart") & (budget_raw["candidate_index"] == 1)]
print("\nold routing rows: repair/B=1 =", len(old_repair_b1), "(expect 24*5=120);",
      "restart/k=1 =", len(old_restart_b1), "(expect 24*5=120)")
assert len(old_repair_b1) == 120 and len(old_restart_b1) == 120
assert sorted(old_repair_b1["routing_seed"].unique().tolist()) == OLD_ROUTING_SEEDS
assert sorted(old_restart_b1["routing_seed"].unique().tolist()) == OLD_ROUTING_SEEDS
# selected repair id must equal the routed B=1 repair id on every seed
sel_ids = set(zip(selected_repairs["circuit"], selected_repairs["seed_plan_id"],
                  selected_repairs["candidate_plan_id"]))
old_ids = set(zip(old_repair_b1["circuit"], old_repair_b1["seed_plan_id"],
                  old_repair_b1["candidate_label"]))
assert sel_ids == old_ids, "Routed B=1 repairs must be exactly the structurally selected repairs."

b1_summary = budget_summary[(budget_summary["B"] == 1)]
print("\nB=1 matched cohort (old budget summary):")
print(b1_summary[["strategy", "n_B"]].to_string(index=False))
print("\nStep 0 complete: 24 frozen B=1 starting cases identified;",
      "24 rank-1 repairs; 24 k=1 restarts; old 5-seed routing matched.")


=== Saved-file inventory (all pre-existing, none modified) ===
experiment1_raw.csv: shape=(250, 13)
experiment1_plan_summary.csv: shape=(50, 17)
experiment2_plan_features.csv: shape=(50, 41)
experiment3_generalized_candidates.csv: shape=(65, 27)
experiment3_restart_generation.csv: shape=(120, 14)
experiment3_budget_raw_routing.csv: shape=(820, 16)
experiment3_budget_summary.csv: shape=(12, 21)

--- key column sets ---
exp1_raw: ['circuit', 'plan_id', 'gidnet_seed', 'plan_signature', 'width', 'raw_logical_depth', 'normalized_logical_depth', 'measurements', 'conditional_x', 'routing_seed', 'swap_count', 'routed_depth', 'cx_count']
exp2_plans (first 12): ['circuit', 'plan_id', 'gidnet_seed', 'width', 'num_reuse_boundaries', 'normalized_logical_depth', 'mean_swaps', 'mean_routed_depth', 'quotient_unique_edges', 'quotient_density', 'max_degree', 'max_weighted_degree'] ...
gen_candidates: ['circuit', 'seed_plan_id', 'candidate_plan_id', 'move_type', 'candidate_chains', 'hotspot_chain_idx', '

## Step 1 — Frozen B=1 evaluation ledger (no fresh data used)

One row per starting case. The Repair is the SAME repair selected in the original
Experiment 3 (`selected == True`, `repair_rank == 1`); the Restart-1 is the SAME restart
(`candidate_index == 1`). Nothing here depends on fresh routing.


In [3]:
# ------------------------------------------------------------
# STEP 1 — frozen B=1 ledger (candidate selection frozen BEFORE fresh eval)
# ------------------------------------------------------------
def parse_saved_structure(value):
    if isinstance(value, (list, tuple, dict)):
        return value
    if pd.isna(value):
        return None
    return ast.literal_eval(str(value))

def canonical_plan_signature(chains):
    return tuple(sorted((tuple(chain) for chain in chains)))

sel = selected_repairs.sort_values(["circuit", "seed_plan_id"]).reset_index(drop=True)
rst = restart_b1.sort_values(["circuit", "seed_plan_id"]).reset_index(drop=True)
assert list(zip(sel["circuit"], sel["seed_plan_id"])) == list(zip(rst["circuit"], rst["seed_plan_id"]))

ledger_rows = []
for i in range(len(sel)):
    s = sel.iloc[i]
    r = rst.iloc[i]
    circuit, seed = s["circuit"], s["seed_plan_id"]
    assert (r["circuit"], r["seed_plan_id"]) == (circuit, seed)
    e = exp2_plans[(exp2_plans["circuit"] == circuit) & (exp2_plans["plan_id"] == seed)].iloc[0]
    orig_chains = [tuple(c) for c in parse_saved_structure(e["reuse_chains"])]
    rep_chains = [tuple(c) for c in parse_saved_structure(s["candidate_chains"])]
    rst_chains = [tuple(c) for c in parse_saved_structure(r["restart_signature"])]
    ledger_rows.append({
        "case_id": f"{circuit}::{seed}",
        "circuit": circuit,
        "starting_plan_id": seed,
        "starting_plan_signature": str(canonical_plan_signature(orig_chains)),
        "original_plan_id": seed,
        "original_plan_signature": str(canonical_plan_signature(orig_chains)),
        "original_gidnet_seed": int(e["gidnet_seed"]),
        "repair_candidate_id": s["candidate_plan_id"],
        "repair_plan_signature": str(canonical_plan_signature(rep_chains)),
        "repair_move_type": s["move_type"],
        "repair_rank": float(s["repair_rank"]),
        "restart_plan_id": f"restart1_g{int(r['restart_gidnet_seed'])}",
        "restart_plan_signature": str(canonical_plan_signature(rst_chains)),
        "restart_gidnet_seed": int(r["restart_gidnet_seed"]),
        "first_restart_seed": int(r["first_restart_seed"]),
        "width": int(e["width"]),
        "original_normalized_depth": int(e["normalized_logical_depth"]),
        "repair_logical_depth_before": int(s["logical_depth_before"]),
        "repair_logical_depth_after": int(s["logical_depth_after"]),
        "restart_logical_depth": int(r["restart_logical_depth"]),
        "same_plan_as_start": bool(r["same_plan_as_start"]),
        "num_generation_attempts": int(r["num_generation_attempts"]),
        # old structural scores that FIXED the repair choice (pre-routing):
        "struct_max_before": float(s["max_weighted_degree_before"]),
        "struct_max_after": float(s["max_weighted_degree_after"]),
        "struct_cv_before": float(s["weighted_degree_cv_before"]),
        "struct_cv_after": float(s["weighted_degree_cv_after"]),
    })

b1_frozen_ledger = pd.DataFrame(ledger_rows)
assert len(b1_frozen_ledger) == 24 and b1_frozen_ledger["case_id"].nunique() == 24
assert (b1_frozen_ledger["repair_rank"] == 1.0).all()

print(f"Frozen B=1 ledger: {len(b1_frozen_ledger)} starting cases (authoritative count = 24).")
print("Circuits covered:", b1_frozen_ledger["circuit"].value_counts().to_dict())
print("Repair move types:", b1_frozen_ledger["repair_move_type"].value_counts().to_dict())
display(b1_frozen_ledger[["case_id", "repair_candidate_id", "repair_move_type",
                          "restart_plan_id", "width", "original_normalized_depth",
                          "repair_logical_depth_after", "restart_logical_depth"]])


Frozen B=1 ledger: 24 starting cases (authoritative count = 24).
Circuits covered: {'mini_alu_305.qasm': 12, 'sym9_146.qasm': 5, 'system9.qasm': 4, 'alu-bdd_288.qasm': 2, 'rd73_140.qasm': 1}
Repair move types: {'one_sided_suffix_transfer': 20, 'two_sided_suffix_swap': 4}


,case_id,repair_candidate_id,repair_move_type,restart_plan_id,width,original_normalized_depth,repair_logical_depth_after,restart_logical_depth
0,alu-bdd_288.qasm::P0,P0_G1,one_sided_suffix_transfer,restart1_g10000,5,51,50,50
1,alu-bdd_288.qasm::P5,P5_G0,one_sided_suffix_transfer,restart1_g10100,5,51,52,50
2,mini_alu_305.qasm::P1,P1_G1,two_sided_suffix_swap,restart1_g10200,6,103,103,94
3,mini_alu_305.qasm::P11,P11_G1,two_sided_suffix_swap,restart1_g10300,6,102,102,95
4,mini_alu_305.qasm::P12,P12_G1,one_sided_suffix_transfer,restart1_g10400,6,103,103,105
5,mini_alu_305.qasm::P13,P13_G1,one_sided_suffix_transfer,restart1_g10500,6,98,101,101
6,mini_alu_305.qasm::P14,P14_G0,one_sided_suffix_transfer,restart1_g10600,6,98,101,103
7,mini_alu_305.qasm::P2,P2_G1,one_sided_suffix_transfer,restart1_g10700,6,94,94,95
8,mini_alu_305.qasm::P3,P3_G1,two_sided_suffix_swap,restart1_g10800,6,103,103,103
9,mini_alu_305.qasm::P4,P4_G1,one_sided_suffix_transfer,restart1_g10900,6,95,95,103


## Reconstruction helpers (exact copies)

The cell below contains **verbatim** copies of the preprocessing / reconstruction /
normalization functions from `experiment3_targeted_hotspot_repair.ipynb`
(parse, canonical signature, boundaries, singleton expansion, full structural validator,
quotient-load metrics, reset normalization, fixed-plan compilation, restart generation).
Only notebook-demo lines that referenced one-off example variables were removed;
every function body is byte-identical in behavior.


In [4]:
# ------------------------------------------------------------
# Reconstruction helpers: VERBATIM copies from experiment3_targeted_hotspot_repair.ipynb
# (cells 6, 7, 18 [canonical signature], 11, 12, 14, 22, 23, 69, 70).
# Only example-specific demo lines were removed; every function body is unchanged.
# ------------------------------------------------------------
from qiskit import qasm2
from gidnet.qubitreuse import GidNET

# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 6 =====
# ------------------------------------------------------------
# Parse reuse-plan representation saved by Experiment 2
# ------------------------------------------------------------

def parse_saved_structure(value):
    if isinstance(value, (list, tuple, dict)):
        return value

    if pd.isna(value):
        return None

    return ast.literal_eval(str(value))

# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 7 =====
# ------------------------------------------------------------
# Verify that saved reuse_boundaries agree with reuse_chains
# ------------------------------------------------------------

def boundaries_from_chains(chains):
    return [
        (chain[i], chain[i + 1])
        for chain in chains
        for i in range(len(chain) - 1)
    ]

# ===== canonical_plan_signature from old notebook cell 18 =====
def canonical_plan_signature(chains):
    """
    Used only for deduplication.
    Internal order of each chain is preserved.
    Chain ordering itself is canonicalized.
    """

    return tuple(
        sorted(
            (tuple(chain) for chain in chains)
        )
    )

# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 11 =====
# ------------------------------------------------------------
# Expand an active reuse plan with inactive singleton qubits
# ------------------------------------------------------------

def expand_with_inactive_singletons(circuit, active_chains):
    used = {
        q
        for chain in active_chains
        for q in chain
    }

    inactive = [
        q for q in range(circuit.num_qubits)
        if q not in used
    ]

    full_chains = [
        list(chain)
        for chain in active_chains
    ]

    full_chains.extend([[q] for q in inactive])

    return full_chains

# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 12 =====
# ------------------------------------------------------------
# Full structural validator for an ACTIVE reuse plan
# ------------------------------------------------------------

import copy
from qiskit.dagcircuit.dagnode import DAGOpNode


def validate_active_plan(circuit, active_chains):
    """
    Validate an active reuse plan against the original circuit.

    active_chains:
        only chains containing active logical qubits,
        exactly as stored by Experiments 1/2.
    """

    # --------------------------------------------------------
    # 1. Find active logical qubits
    # --------------------------------------------------------

    active = set()

    for item in circuit.data:
        if item.operation.name in {"barrier", "measure"}:
            continue

        for q in item.qubits:
            active.add(
                circuit.find_bit(q).index
            )

    # --------------------------------------------------------
    # 2. Every active qubit must appear exactly once
    # --------------------------------------------------------

    flat = [
        q
        for chain in active_chains
        for q in chain
    ]

    if len(flat) != len(set(flat)):
        return {
            "legal": False,
            "reason": "duplicate_logical_qubit",
        }

    if set(flat) != active:
        return {
            "legal": False,
            "reason": "active_qubit_set_mismatch",
            "missing": sorted(active - set(flat)),
            "extra": sorted(set(flat) - active),
        }

    # --------------------------------------------------------
    # 3. Add inactive declared qubits as singleton chains
    # --------------------------------------------------------

    full_chains = expand_with_inactive_singletons(
        circuit,
        active_chains
    )

    # --------------------------------------------------------
    # 4. Fresh GidNET candidate matrix
    # --------------------------------------------------------

    test_gid = GidNET(
        copy.deepcopy(circuit)
    )

    C = np.array(
        test_gid.candidate_matrix_copy,
        copy=True
    )

    boundaries = boundaries_from_chains(
        active_chains
    )

    illegal_boundaries = []

    for a, b in boundaries:
        if C[a, b] != 1:
            illegal_boundaries.append((a, b))

    if illegal_boundaries:
        return {
            "legal": False,
            "reason": "boundary_not_in_candidate_matrix",
            "illegal_boundaries": illegal_boundaries,
        }

    # --------------------------------------------------------
    # 5. Apply COMPLETE plan using GidNET's own DAG logic
    # --------------------------------------------------------

    test_gid.qubit_reuse_sequences = copy.deepcopy(
        full_chains
    )

    try:
        test_gid.add_qubit_reuse_edges()

    except Exception as e:
        return {
            "legal": False,
            "reason": "add_reuse_edges_failed",
            "error": repr(e),
        }

    # Same barrier handling as compile_to_dynamic_circuit()
    barrier_nodes = [
        node
        for node in list(
            test_gid.circuit_dag_with_reuse_edges.nodes()
        )
        if isinstance(node, DAGOpNode)
        and node.op.name == "barrier"
    ]

    for node in barrier_nodes:
        test_gid.circuit_dag_with_reuse_edges.remove_op_node(
            node
        )

    # --------------------------------------------------------
    # 6. Full DAG-cycle check
    # --------------------------------------------------------

    try:
        all_nodes = list(
            test_gid.circuit_dag_with_reuse_edges.nodes()
        )

        topo_nodes = list(
            test_gid.circuit_dag_with_reuse_edges.topological_nodes()
        )

    except Exception as e:
        return {
            "legal": False,
            "reason": "topological_sort_failed",
            "error": repr(e),
        }

    if len(all_nodes) != len(topo_nodes):
        return {
            "legal": False,
            "reason": "dependency_cycle",
        }

    return {
        "legal": True,
        "reason": "ok",
        "active_width": len(active_chains),
        "full_chain_count": len(full_chains),
        "num_reuse_boundaries": len(boundaries),
    }
# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 14 =====
# ------------------------------------------------------------
# Reproduce Experiment-2 quotient interaction-load metrics
# ------------------------------------------------------------

from collections import defaultdict


def compute_quotient_load_metrics(circuit, chains):
    """
    Build the quotient interaction graph induced by reuse chains.

    Each reuse chain is one quotient node.
    Every original two-qubit interaction between logical qubits
    belonging to different chains increments the corresponding
    quotient-edge weight by one.
    """

    # logical qubit -> reuse-chain index
    qubit_to_chain = {}

    for chain_idx, chain in enumerate(chains):
        for q in chain:
            if q in qubit_to_chain:
                raise ValueError(
                    f"Logical qubit {q} appears in multiple chains."
                )

            qubit_to_chain[q] = chain_idx

    # Weighted undirected quotient edges
    edge_weights = defaultdict(int)

    for item in circuit.data:

        qargs = list(item.qubits)

        # Only two-qubit interactions
        if len(qargs) != 2:
            continue

        q0 = circuit.find_bit(qargs[0]).index
        q1 = circuit.find_bit(qargs[1]).index

        # Ignore declared-but-inactive qubits not represented
        # in the active reuse plan.
        if (
            q0 not in qubit_to_chain
            or q1 not in qubit_to_chain
        ):
            continue

        c0 = qubit_to_chain[q0]
        c1 = qubit_to_chain[q1]

        # Interaction inside the same reused wire does not
        # create an inter-chain quotient edge.
        if c0 == c1:
            continue

        edge = tuple(sorted((c0, c1)))
        edge_weights[edge] += 1

    # --------------------------------------------------------
    # Weighted degree of every quotient node
    # --------------------------------------------------------

    weighted_degree = np.zeros(
        len(chains),
        dtype=float
    )

    for (u, v), weight in edge_weights.items():
        weighted_degree[u] += weight
        weighted_degree[v] += weight

    max_weighted_degree = (
        float(weighted_degree.max())
        if len(weighted_degree)
        else 0.0
    )

    mean_wd = float(weighted_degree.mean())

    weighted_degree_cv = (
        float(weighted_degree.std(ddof=0) / mean_wd)
        if mean_wd > 0
        else 0.0
    )

    hotspot_indices = [
        i
        for i, value in enumerate(weighted_degree)
        if np.isclose(
            value,
            max_weighted_degree
        )
    ]

    return {
        "edge_weights": dict(edge_weights),
        "weighted_degree": weighted_degree,
        "max_weighted_degree": max_weighted_degree,
        "weighted_degree_cv": weighted_degree_cv,
        "hotspot_indices": hotspot_indices,
    }
# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 22 =====
from qiskit import QuantumCircuit, ClassicalRegister


def normalize_active_reset(qc: QuantumCircuit) -> QuantumCircuit:

    new_qc = QuantumCircuit(*qc.qregs, *qc.cregs)

    last_measure_clbit = {}
    clbit_owner = {}

    total_resets = sum(
        1
        for inst in qc.data
        if inst.operation.name == "reset"
    )

    aux_reg = None
    aux_next = 0

    for item in qc.data:

        instr = item.operation
        qargs = list(item.qubits)
        cargs = list(item.clbits)

        q_indices = [
            qc.find_bit(q).index
            for q in qargs
        ]

        # -----------------------------
        # Measurement
        # -----------------------------

        if instr.name == "measure":

            q = qargs[0]
            c = cargs[0]

            q_idx = qc.find_bit(q).index
            c_idx = qc.find_bit(c).index

            old_owner = clbit_owner.get(c_idx)

            if old_owner is not None:
                last_measure_clbit.pop(
                    old_owner,
                    None
                )

            new_qc.append(
                instr,
                qargs,
                cargs
            )

            new_cbit = new_qc.clbits[c_idx]

            last_measure_clbit[q_idx] = (
                new_cbit
            )

            clbit_owner[c_idx] = q_idx

            continue

        # -----------------------------
        # Reset
        # -----------------------------

        if instr.name == "reset":

            q = qargs[0]

            q_idx = qc.find_bit(q).index

            if q_idx in last_measure_clbit:

                cbit = (
                    last_measure_clbit[q_idx]
                )

            else:

                if aux_reg is None:

                    aux_reg = ClassicalRegister(
                        max(total_resets, 1),
                        "reset_aux"
                    )

                    new_qc.add_register(
                        aux_reg
                    )

                cbit = aux_reg[aux_next]

                aux_next += 1

                new_qc.measure(
                    q,
                    cbit
                )

            new_qc.x(q).c_if(
                cbit,
                1
            )

            last_measure_clbit.pop(
                q_idx,
                None
            )

            continue

        # -----------------------------
        # Other operation
        # -----------------------------

        new_qc.append(
            instr,
            qargs,
            cargs
        )

        for q_idx in q_indices:

            last_measure_clbit.pop(
                q_idx,
                None
            )

    return new_qc
# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 23 =====
# ------------------------------------------------------------
# Compile a FIXED reuse plan through GidNET's own pipeline
# ------------------------------------------------------------

def get_active_qubit_count(qc):

    active = set()

    for item in qc.data:

        for q in item.qubits:

            active.add(
                qc.find_bit(q).index
            )

    return len(active)


def compile_fixed_active_plan(
    circuit,
    active_chains
):
    """
    Compile exactly the supplied active reuse plan using
    GidNET's existing dynamic-circuit construction.

    GidNET's sequence-selection step is replaced ONLY for
    this instance; the rest of the compilation pipeline is
    unchanged.
    """

    full_chains = (
        expand_with_inactive_singletons(
            circuit,
            active_chains
        )
    )

    g = GidNET(
        copy.deepcopy(circuit)
    )

    # Force GidNET to use exactly this plan.
    g.compute_optimized_reuse_sequences = (
        lambda iterations:
            copy.deepcopy(full_chains)
    )

    dynamic_circuit = (
        g.compile_to_dynamic_circuit(
            iterations=1,
            draw=False
        )
    )

    return dynamic_circuit, g
# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 69 =====
# ------------------------------------------------------------
# Helpers for independent GidNET restarts
# ------------------------------------------------------------

import time


def get_original_active_qubit_indices(circuit):
    """
    Same active-qubit interpretation used earlier:
    ignore barriers and measurements when determining
    the logical qubits participating in the computation.
    """

    active = set()

    for item in circuit.data:

        if item.operation.name in {
            "barrier",
            "measure",
        }:
            continue

        for q in item.qubits:
            active.add(
                circuit.find_bit(q).index
            )

    return sorted(active)


def extract_active_reuse_chains(
    circuit,
    raw_reuse_sequences,
):
    """
    GidNET may contain declared-but-inactive qubits
    (and, in this implementation, additional matrix indices).

    Keep only original ACTIVE logical-qubit indices and
    remove empty chains.
    """

    active_set = set(
        get_original_active_qubit_indices(
            circuit
        )
    )

    active_chains = []

    for chain in raw_reuse_sequences:

        filtered = tuple(
            q
            for q in chain
            if q in active_set
        )

        if filtered:
            active_chains.append(
                filtered
            )

    # Sanity: every active logical qubit exactly once
    flat = [
        q
        for chain in active_chains
        for q in chain
    ]

    if (
        len(flat) != len(set(flat))
        or
        set(flat) != active_set
    ):
        raise ValueError(
            "Restart plan does not represent every "
            "active logical qubit exactly once."
        )

    return active_chains
# ===== EXACT SOURCE OF OLD NOTEBOOK CELL 70 =====
# ------------------------------------------------------------
# Generate ONE independent same-width GidNET restart
# ------------------------------------------------------------

def generate_same_width_restart(
    circuit,
    target_width,
    first_seed,
    max_attempts=50,
):
    """
    Generate one GidNET restart whose ACTIVE reuse width
    equals target_width.

    Width-miss attempts are recorded but not physically routed.
    The first same-width result is accepted even if it reproduces
    the original plan.
    """

    attempts = []

    total_start = time.perf_counter()

    for offset in range(max_attempts):

        gidnet_seed = (
            first_seed + offset
        )

        random.seed(
            gidnet_seed
        )

        np.random.seed(
            gidnet_seed
        )

        t0 = time.perf_counter()

        g = GidNET(
            copy.deepcopy(circuit)
        )

        dynamic = (
            g.compile_to_dynamic_circuit(
                iterations=20,
                draw=False,
            )
        )

        runtime = (
            time.perf_counter()
            - t0
        )

        active_chains = (
            extract_active_reuse_chains(
                circuit,
                g.qubit_reuse_sequences,
            )
        )

        active_width = len(
            active_chains
        )

        attempts.append({
            "gidnet_seed":
                gidnet_seed,

            "active_width":
                active_width,

            "generation_runtime_sec":
                runtime,
        })

        if active_width != target_width:
            continue

        dynamic_norm = (
            normalize_active_reset(
                dynamic
            )
        )

        return {
            "success":
                True,

            "gidnet_seed":
                gidnet_seed,

            "active_chains":
                active_chains,

            "plan_signature":
                canonical_plan_signature(
                    active_chains
                ),

            "active_width":
                active_width,

            "normalized_circuit":
                dynamic_norm,

            "normalized_logical_depth":
                dynamic_norm.depth(),

            "num_generation_attempts":
                len(attempts),

            "total_generation_runtime_sec":
                time.perf_counter()
                - total_start,

            "attempts":
                attempts,
        }

    return {
        "success":
            False,

        "num_generation_attempts":
            len(attempts),

        "total_generation_runtime_sec":
            time.perf_counter()
            - total_start,

        "attempts":
            attempts,
    }
# ------------------------------------------------------------
# Benchmark circuits + per-circuit cache (same pattern as old notebook cell 21)
# ------------------------------------------------------------
BENCH_DIR = Path(
    r"C:\Users\ASUS\Documents\Quantum computing\Qiskit"
    r"\CaQR-main\CaQR-main\benchmarks"
)
assert BENCH_DIR.exists(), f"Benchmark directory missing: {BENCH_DIR}"
circuit_cache = {}
def get_benchmark_circuit(circuit_name):
    if circuit_name not in circuit_cache:
        circuit_path = BENCH_DIR / circuit_name
        if not circuit_path.exists():
            raise FileNotFoundError(f"Missing benchmark: {circuit_path}")
        circuit_cache[circuit_name] = qasm2.load(str(circuit_path))
    return circuit_cache[circuit_name]
print("Reconstruction helpers loaded (verbatim from original Experiment 3).")
print("Benchmark directory:", BENCH_DIR)


Reconstruction helpers loaded (verbatim from original Experiment 3).
Benchmark directory: C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks


## Step 2 — Reconstruct and verify ALL fixed alternatives (checkpoint gate)

- **Original / Repair**: `compile_fixed_active_plan` on the stored active chains.
- **Restart-1**: deterministic regeneration with `generate_same_width_restart`
  using the stored `first_restart_seed` (the restart's own dynamic circuit came from a
  real GidNET run, so only exact regeneration reproduces it bit-identically; this is the
  same resume logic the original notebook uses for k>1 restarts).

Every alternative must satisfy: canonical signature match, width match, full legality
(`validate_active_plan`), and normalized-depth match. **Any failure stops the notebook**
— nothing is silently replaced. Only after `24/24 × 3` pass may the routing cell run.


In [5]:
# ------------------------------------------------------------
# STEP 2 — reconstruct + verify all 72 fixed alternatives (NO routing here)
# ------------------------------------------------------------
verification_rows = []
verification_failures = []
# Normalized circuits retained for the fresh-routing cell (same session, in order).
FRESH_NORM_CIRCUITS = {}
FRESH_PLAN_META = {}

for _, led in b1_frozen_ledger.iterrows():
    case_id = led["case_id"]
    circuit, seed = led["circuit"], led["starting_plan_id"]
    qc = get_benchmark_circuit(circuit)

    orig_chains = [tuple(c) for c in parse_saved_structure(
        exp2_plans[(exp2_plans["circuit"] == circuit)
                   & (exp2_plans["plan_id"] == seed)].iloc[0]["reuse_chains"])]
    rep_row = sel[(sel["circuit"] == circuit) & (sel["seed_plan_id"] == seed)].iloc[0]
    rep_chains = [tuple(c) for c in parse_saved_structure(rep_row["candidate_chains"])]

    plans = [
        ("Original", led["original_plan_id"], orig_chains,
         led["original_plan_signature"], int(led["width"]),
         int(led["original_normalized_depth"]), None),
        ("Repair", led["repair_candidate_id"], rep_chains,
         led["repair_plan_signature"], int(led["width"]),
         int(led["repair_logical_depth_after"]), None),
        ("Restart-1", led["restart_plan_id"], None,
         led["restart_plan_signature"], int(led["width"]),
         int(led["restart_logical_depth"]),
         {"first_seed": int(led["first_restart_seed"]),
          "gidnet_seed": int(led["restart_gidnet_seed"])}),
    ]

    for alternative, plan_id, chains, stored_sig_str, exp_width, exp_depth, restart_info in plans:
        stored_sig = parse_saved_structure(stored_sig_str)
        stored_sig = canonical_plan_signature([tuple(c) for c in stored_sig])
        rec = {"case_id": case_id, "circuit": circuit, "starting_plan_id": seed,
               "alternative": alternative, "plan_id": plan_id, "ok": False, "detail": ""}
        try:
            if alternative == "Restart-1":
                # Exact deterministic regeneration of the stored GidNET restart.
                regen = generate_same_width_restart(
                    circuit=qc, target_width=exp_width,
                    first_seed=restart_info["first_seed"], max_attempts=50)
                assert regen["success"], "restart regeneration failed"
                assert int(regen["gidnet_seed"]) == restart_info["gidnet_seed"], (
                    f"gidnet seed {regen['gidnet_seed']} != stored {restart_info['gidnet_seed']}")
                chains = [tuple(c) for c in regen["active_chains"]]
                dynamic = None  # regen already holds the exact dynamic circuit path
                norm = regen["normalized_circuit"]
                raw_depth = None  # dynamic pre-normalization depth not re-derived; use norm only
                rec["regen_attempts"] = int(regen["num_generation_attempts"])
            else:
                v = validate_active_plan(qc, chains)
                assert v["legal"], f"illegal plan: {v.get('reason')}"
                dynamic, _ = compile_fixed_active_plan(qc, chains)
                assert get_active_qubit_count(dynamic) == exp_width, "width mismatch after compile"
                raw_depth = int(dynamic.depth())
                norm = normalize_active_reset(dynamic)

            sig = canonical_plan_signature(chains)
            assert sig == stored_sig, (
                f"signature mismatch:\n stored={stored_sig}\n rebuilt={sig}")
            assert len(chains) == exp_width, "active width mismatch"
            v = validate_active_plan(qc, chains)
            assert v["legal"], f"illegal plan: {v.get('reason')}"
            norm_depth = int(norm.depth())
            assert norm_depth == exp_depth, (
                f"normalized depth {norm_depth} != stored {exp_depth}")

            FRESH_NORM_CIRCUITS[(case_id, alternative)] = norm
            FRESH_PLAN_META[(case_id, alternative)] = {
                "plan_id": plan_id, "plan_signature": str(sig), "width": exp_width,
                "raw_logical_depth": raw_depth if raw_depth is not None else norm_depth,
                "normalized_logical_depth": norm_depth,
                "measurements": int(norm.count_ops().get("measure", 0)),
            }
            rec["ok"] = True
            rec["detail"] = f"sig ok; width={exp_width}; norm_depth={norm_depth}"
        except AssertionError as e:
            rec["detail"] = f"FAIL: {e}"
            old_sig = stored_sig_str
            new_sig = None
            try:
                new_sig = str(canonical_plan_signature(chains)) if chains else None
            except Exception:
                pass
            verification_failures.append({
                "circuit": circuit, "case_id": case_id, "alternative": alternative,
                "old_signature": old_sig, "reconstructed_signature": new_sig,
                "old_width": exp_width, "reason": str(e)[:300]})
        verification_rows.append(rec)

verification_df = pd.DataFrame(verification_rows)
n_cases = len(b1_frozen_ledger)
ok_orig = int(verification_df[verification_df["alternative"] == "Original"]["ok"].sum())
ok_rep = int(verification_df[verification_df["alternative"] == "Repair"]["ok"].sum())
ok_rst = int(verification_df[verification_df["alternative"] == "Restart-1"]["ok"].sum())
print(f"Verification: Original {ok_orig}/{n_cases}; Repair {ok_rep}/{n_cases}; "
      f"Restart-1 {ok_rst}/{n_cases}")
display(verification_df[["case_id", "alternative", "plan_id", "ok", "detail"]])
if verification_failures:
    print("\nFAILURES (reconstruction stopped, nothing replaced):")
    display(pd.DataFrame(verification_failures))

VERIFIED = (ok_orig == n_cases and ok_rep == n_cases and ok_rst == n_cases
            and len(FRESH_NORM_CIRCUITS) == 3 * n_cases)
assert VERIFIED, ("STOP: not all frozen alternatives reconstructed exactly. "
                  "Resolve the failures above; do NOT run fresh routing.")
print(f"\nAll {n_cases} B=1 cases reconstructed and verified: "
      f"{n_cases} Original, {n_cases} Repair, {n_cases} Restart-1.")
print("Ready for fresh 30-seed evaluation.")


Verification: Original 24/24; Repair 24/24; Restart-1 24/24


,case_id,alternative,plan_id,ok,detail
0,alu-bdd_288.qasm::P0,Original,P0,True,sig ok; width=5; norm_depth=51
1,alu-bdd_288.qasm::P0,Repair,P0_G1,True,sig ok; width=5; norm_depth=50
2,alu-bdd_288.qasm::P0,Restart-1,restart1_g10000,True,sig ok; width=5; norm_depth=50
3,alu-bdd_288.qasm::P5,Original,P5,True,sig ok; width=5; norm_depth=51
4,alu-bdd_288.qasm::P5,Repair,P5_G0,True,sig ok; width=5; norm_depth=52
...,...,...,...,...,...
67,system9.qasm::P2,Repair,P2_G0,True,sig ok; width=7; norm_depth=166
68,system9.qasm::P2,Restart-1,restart1_g12200,True,sig ok; width=7; norm_depth=166
69,system9.qasm::P8,Original,P8,True,sig ok; width=7; norm_depth=177
70,system9.qasm::P8,Repair,P8_G3,True,sig ok; width=7; norm_depth=165



All 24 B=1 cases reconstructed and verified: 24 Original, 24 Repair, 24 Restart-1.
Ready for fresh 30-seed evaluation.


## Step 3 — Physical compiler (exactly as in the original Experiment 3)

Same `Heavy-Hex(7)` coupling map, same `transpile` settings
(`routing_method="sabre"`, `layout_method="sabre"`, `optimization_level=1`),
only the `seed_transpiler` values change (fresh seeds 100–129).


In [6]:
# ------------------------------------------------------------
# STEP 3 — exact physical compiler from old notebook cell 31
# ------------------------------------------------------------
from qiskit import transpile
from qiskit.transpiler import CouplingMap

TARGET_COUPLING = (
    CouplingMap.from_heavy_hex(7)
)

print(
    "Heavy-Hex physical qubits:",
    TARGET_COUPLING.size()
)
assert TARGET_COUPLING.size() == 115

TRANSPILE_KWARGS = dict(
    coupling_map=TARGET_COUPLING,
    routing_method="sabre",
    layout_method="sabre",
    optimization_level=1,
)
print("Transpile settings:", {k: (v if not hasattr(v, "size") else f"CouplingMap({v.size()})")
                              for k, v in TRANSPILE_KWARGS.items()})
print("Fresh routing seeds:", FRESH_ROUTING_SEEDS[0], "..", FRESH_ROUTING_SEEDS[-1],
      f"({len(FRESH_ROUTING_SEEDS)} seeds)")


Heavy-Hex physical qubits: 115
Transpile settings: {'coupling_map': 'CouplingMap(115)', 'routing_method': 'sabre', 'layout_method': 'sabre', 'optimization_level': 1}
Fresh routing seeds: 100 .. 129 (30 seeds)


## Steps 4–5 — Fresh 30-seed routing (robust / resumable)

- `FRESH_ROUTING_SEEDS = 100..129` only; old seeds 42–46 never enter fresh means.
- Per case: Original × 30 + Repair × 30 + Restart-1 × 30 → 24 × 3 × 30 = **2160** runs.
- Qiskit INFO logging is suppressed; only compact progress is printed.
- Results checkpoint to `experiment3_b1_fresh30_partial_checkpoint.csv` after every case;
  reruns skip already completed `(case_id, alternative, routing_seed)` triples.
- **Gate:** this cell refuses to run unless the Step-2 `VERIFIED` flag is set.

NOTE: `raw_logical_depth` is the pre-normalization dynamic-circuit depth for
Original/Repair. Restart-1 dynamic circuits come from exact deterministic regeneration
of the stored GidNET run, so both depth fields equal the verified normalized depth.


In [7]:
# ------------------------------------------------------------
# STEPS 4-5 — fresh 30-seed physical routing (checkpoint-safe)
# ------------------------------------------------------------
assert "VERIFIED" in globals() and VERIFIED, (
    "STOP: run the Step-2 verification cell first and resolve any failure.")
assert len(FRESH_NORM_CIRCUITS) == 3 * len(b1_frozen_ledger)

import logging
logging.disable(logging.INFO)  # suppress Qiskit transpiler INFO spam; re-enabled below.

def routed_metrics(routed):
    ops = routed.count_ops()
    cond_x = 0
    for inst in routed.data:
        try:
            if inst.operation.name == "x" and getattr(inst.operation, "condition", None) is not None:
                cond_x += 1
        except Exception:
            pass
    return (int(ops.get("swap", 0)), int(routed.depth()), int(ops.get("cx", 0)),
            int(ops.get("measure", 0)), int(cond_x))

N_CASES = len(b1_frozen_ledger)
TOTAL_RUNS = N_CASES * 3 * len(FRESH_ROUTING_SEEDS)
print(f"Expecting {N_CASES} x 3 x {len(FRESH_ROUTING_SEEDS)} = {TOTAL_RUNS} routing evaluations.")

done_keys = set()
existing_rows = []
if PARTIAL_PATH.exists():
    chk = pd.read_csv(PARTIAL_PATH)
    for _, r in chk.iterrows():
        done_keys.add((r["case_id"], r["alternative"], int(r["routing_seed"])))
    existing_rows = chk.to_dict("records")
    print(f"Resuming: {len(existing_rows)} completed runs loaded from checkpoint; "
          f"{TOTAL_RUNS - len(existing_rows)} remaining.")

fresh_rows = list(existing_rows)
completed = len(fresh_rows)
run_counter = 0
for _, led in b1_frozen_ledger.iterrows():
    case_id = led["case_id"]
    for alternative in ["Original", "Repair", "Restart-1"]:
        norm = FRESH_NORM_CIRCUITS[(case_id, alternative)]
        meta = FRESH_PLAN_META[(case_id, alternative)]
        for rs in FRESH_ROUTING_SEEDS:
            if (case_id, alternative, rs) in done_keys:
                continue
            routed = transpile(norm, seed_transpiler=rs, **TRANSPILE_KWARGS)
            swaps, depth, cx, meas, condx = routed_metrics(routed)
            fresh_rows.append({
                "case_id": case_id,
                "circuit": led["circuit"],
                "starting_plan_id": led["starting_plan_id"],
                "alternative": alternative,
                "plan_id": meta["plan_id"],
                "plan_signature": meta["plan_signature"],
                "width": meta["width"],
                "raw_logical_depth": meta["raw_logical_depth"],
                "normalized_logical_depth": meta["normalized_logical_depth"],
                "routing_seed": rs,
                "swap_count": swaps,
                "routed_depth": depth,
                "cx_count": cx,
                "measurements": meas,
                "conditional_x": condx,
            })
            done_keys.add((case_id, alternative, rs))
            completed += 1
            run_counter += 1
            if completed % 50 == 0:
                print(f"{completed} / {TOTAL_RUNS}", flush=True)
    # checkpoint after every completed case (crash-safe, idempotent)
    pd.DataFrame(fresh_rows).to_csv(PARTIAL_PATH, index=False)

fresh30_raw = pd.DataFrame(fresh_rows).sort_values(
    ["case_id", "alternative", "routing_seed"]).reset_index(drop=True)
fresh30_raw.to_csv(PARTIAL_PATH, index=False)
logging.disable(logging.NOTSET)  # re-enable logging
print(f"\nFresh routing complete: {len(fresh30_raw)} / {TOTAL_RUNS} runs.")
assert len(fresh30_raw) == TOTAL_RUNS
cov = fresh30_raw.groupby(["case_id", "alternative"])["routing_seed"].nunique()
assert (cov == len(FRESH_ROUTING_SEEDS)).all(), "Every (case, alternative) needs exactly 30 seeds."
print("Coverage ok: every (case_id, alternative) has exactly 30 distinct fresh seeds.")


Expecting 24 x 3 x 30 = 2160 routing evaluations.
50 / 2160
100 / 2160
150 / 2160
200 / 2160
250 / 2160
300 / 2160
350 / 2160
400 / 2160
450 / 2160
500 / 2160
550 / 2160
600 / 2160
650 / 2160
700 / 2160
750 / 2160
800 / 2160
850 / 2160
900 / 2160
950 / 2160
1000 / 2160
1050 / 2160
1100 / 2160
1150 / 2160
1200 / 2160
1250 / 2160
1300 / 2160
1350 / 2160
1400 / 2160
1450 / 2160
1500 / 2160
1550 / 2160
1600 / 2160
1650 / 2160
1700 / 2160
1750 / 2160
1800 / 2160
1850 / 2160
1900 / 2160
1950 / 2160
2000 / 2160
2050 / 2160
2100 / 2160
2150 / 2160

Fresh routing complete: 2160 / 2160 runs.
Coverage ok: every (case_id, alternative) has exactly 30 distinct fresh seeds.


## Step 6 — Raw fresh results (`fresh30_raw`)

One row per physical routing evaluation. The asserts below enforce
`N × 3 × 30` rows with exactly 30 distinct fresh seeds per `(case_id, alternative)`,
and confirm no old seed (42–46) leaked into the fresh table.


In [8]:
# ------------------------------------------------------------
# STEP 6 — raw-result integrity checks
# ------------------------------------------------------------
N = len(b1_frozen_ledger)
assert len(fresh30_raw) == N * 3 * len(FRESH_ROUTING_SEEDS), (
    f"len={len(fresh30_raw)} != {N * 3 * len(FRESH_ROUTING_SEEDS)}")
assert set(fresh30_raw["alternative"].unique().tolist()) == {"Original", "Repair", "Restart-1"}
assert not set(fresh30_raw["routing_seed"].unique()) & set(OLD_ROUTING_SEEDS), (
    "Old routing seeds must never appear in fresh means.")
assert set(fresh30_raw["routing_seed"].unique()) == set(FRESH_ROUTING_SEEDS)
per_alt = fresh30_raw.groupby(["case_id", "alternative"])["routing_seed"].agg(["nunique", "count"])
assert ((per_alt["nunique"] == 30) & (per_alt["count"] == 30)).all()
print(f"fresh30_raw: {len(fresh30_raw)} rows = {N} cases x 3 alternatives x 30 seeds. OK.")
display(fresh30_raw.head(10))
print(fresh30_raw.groupby("alternative")[["swap_count", "routed_depth"]].describe().round(2).to_string())


fresh30_raw: 2160 rows = 24 cases x 3 alternatives x 30 seeds. OK.


,case_id,circuit,starting_plan_id,alternative,plan_id,plan_signature,width,raw_logical_depth,normalized_logical_depth,routing_seed,swap_count,routed_depth,cx_count,measurements,conditional_x
0,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,100,21,81,38,2,2
1,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,101,23,81,38,2,2
2,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,102,21,80,38,2,2
3,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,103,23,82,38,2,2
4,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,104,20,79,38,2,2
5,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,105,23,82,38,2,2
6,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,106,21,82,38,2,2
7,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,107,21,82,38,2,2
8,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,108,23,81,38,2,2
9,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,49,51,109,20,80,38,2,2


            swap_count                                               routed_depth                                                  
                 count   mean    std   min   25%   50%    75%    max        count    mean    std   min    25%    50%     75%    max
alternative                                                                                                                        
Original         720.0  59.39  23.61  20.0  44.0  46.0  87.00  107.0        720.0  210.84  80.77  79.0  157.0  162.0  304.25  332.0
Repair           720.0  58.72  23.24  20.0  43.0  46.0  85.00  107.0        720.0  208.25  79.27  76.0  156.0  163.0  298.00  337.0
Restart-1        720.0  57.90  22.61  21.0  43.0  45.0  84.25  101.0        720.0  210.08  80.55  77.0  157.0  161.0  302.00  337.0


## Step 7 — Plan / alternative-level fresh summary

Per `(case_id, alternative)`: fresh mean / std / min / max of SWAP and routed depth,
plus the number of fresh routing trials.


In [9]:
# ------------------------------------------------------------
# STEP 7 — alternative-level fresh summary
# ------------------------------------------------------------
fresh30_alternative_summary = (
    fresh30_raw.groupby(["case_id", "circuit", "starting_plan_id",
                         "alternative", "plan_id", "plan_signature", "width"])
    .agg(fresh_mean_swaps=("swap_count", "mean"),
         fresh_std_swaps=("swap_count", "std"),
         fresh_min_swaps=("swap_count", "min"),
         fresh_max_swaps=("swap_count", "max"),
         fresh_mean_routed_depth=("routed_depth", "mean"),
         fresh_std_routed_depth=("routed_depth", "std"),
         fresh_min_routed_depth=("routed_depth", "min"),
         fresh_max_routed_depth=("routed_depth", "max"),
         num_fresh_routing_trials=("routing_seed", "size"))
    .reset_index()
)
assert len(fresh30_alternative_summary) == N * 3
assert (fresh30_alternative_summary["num_fresh_routing_trials"] == 30).all()
print(f"fresh30_alternative_summary: {len(fresh30_alternative_summary)} rows (24 x 3).")
display(fresh30_alternative_summary.head(12))


fresh30_alternative_summary: 72 rows (24 x 3).


,case_id,circuit,starting_plan_id,alternative,plan_id,plan_signature,width,fresh_mean_swaps,fresh_std_swaps,fresh_min_swaps,fresh_max_swaps,fresh_mean_routed_depth,fresh_std_routed_depth,fresh_min_routed_depth,fresh_max_routed_depth,num_fresh_routing_trials
0,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Original,P0,"((0,), (1, 6), (2,), (3, 4), (5,))",5,21.066667,1.172481,20,23,80.466667,1.008014,79,82,30
1,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Repair,P0_G1,"((0,), (1, 6), (2, 4), (3,), (5,))",5,22.033333,1.299425,20,24,77.933333,1.257620,76,82,30
2,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,P0,Restart-1,restart1_g10000,"((0,), (1, 2, 4), (3,), (5,), (6,))",5,23.433333,1.546594,21,25,79.000000,1.761661,77,82,30
3,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,P5,Original,P5,"((0,), (1, 2), (3, 4), (5,), (6,))",5,22.266667,1.172481,21,24,82.633333,1.272612,80,85,30
4,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,P5,Repair,P5_G0,"((0, 4), (1, 2), (3,), (5,), (6,))",5,21.300000,0.466092,21,22,80.533333,2.112905,77,83,30
5,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,P5,Restart-1,restart1_g10100,"((0,), (1, 2, 4), (3,), (5,), (6,))",5,23.433333,1.546594,21,25,79.000000,1.761661,77,82,30
6,mini_alu_305.qasm::P1,mini_alu_305.qasm,P1,Original,P1,"((0,), (1, 3), (2, 5, 8), (4,), (6, 9), (7,))",6,42.766667,1.278019,42,48,158.800000,1.954658,156,166,30
7,mini_alu_305.qasm::P1,mini_alu_305.qasm,P1,Repair,P1_G1,"((0,), (1, 3), (2, 5, 9), (4,), (6, 8), (7,))",6,43.900000,1.155198,43,48,158.766667,1.612095,156,163,30
8,mini_alu_305.qasm::P1,mini_alu_305.qasm,P1,Restart-1,restart1_g10200,"((1, 5, 9), (2, 8), (3,), (4,), (6,), (7, 0))",6,41.966667,1.902509,40,46,152.566667,6.739914,139,160,30
9,mini_alu_305.qasm::P11,mini_alu_305.qasm,P11,Original,P11,"((1, 5, 8), (2, 0), (3,), (4,), (6, 9), (7,))",6,44.766667,0.727932,44,47,158.700000,2.830255,155,162,30


## Step 8 — Case-level improvements vs Original

Improvement is defined so **positive = better** (Original mean minus alternative mean),
with percentage versions relative to the Original fresh mean.


In [10]:
# ------------------------------------------------------------
# STEP 8 — case-level Repair/Restart vs Original (fresh means)
# ------------------------------------------------------------
piv = fresh30_alternative_summary.pivot(
    index=["case_id", "circuit"], columns="alternative",
    values=["fresh_mean_swaps", "fresh_mean_routed_depth"])
piv.columns = [f"{a}__{m}" for m, a in piv.columns]
piv = piv.reset_index()

def _pct(orig, alt):
    return np.where(np.asarray(orig, dtype=float) != 0,
                    100.0 * (np.asarray(orig, dtype=float) - np.asarray(alt, dtype=float))
                    / np.asarray(orig, dtype=float), np.nan)

case_rows = pd.DataFrame({
    "case_id": piv["case_id"],
    "circuit": piv["circuit"],
    "original_mean_swaps": piv["Original__fresh_mean_swaps"],
    "repair_mean_swaps": piv["Repair__fresh_mean_swaps"],
    "restart_mean_swaps": piv["Restart-1__fresh_mean_swaps"],
    "original_mean_depth": piv["Original__fresh_mean_routed_depth"],
    "repair_mean_depth": piv["Repair__fresh_mean_routed_depth"],
    "restart_mean_depth": piv["Restart-1__fresh_mean_routed_depth"],
})
case_rows["repair_swap_improvement"] = (
    case_rows["original_mean_swaps"] - case_rows["repair_mean_swaps"])
case_rows["restart_swap_improvement"] = (
    case_rows["original_mean_swaps"] - case_rows["restart_mean_swaps"])
case_rows["repair_swap_improvement_pct"] = _pct(
    case_rows["original_mean_swaps"], case_rows["repair_mean_swaps"])
case_rows["restart_swap_improvement_pct"] = _pct(
    case_rows["original_mean_swaps"], case_rows["restart_mean_swaps"])
case_rows["repair_depth_improvement"] = (
    case_rows["original_mean_depth"] - case_rows["repair_mean_depth"])
case_rows["restart_depth_improvement"] = (
    case_rows["original_mean_depth"] - case_rows["restart_mean_depth"])
case_rows["repair_depth_improvement_pct"] = _pct(
    case_rows["original_mean_depth"], case_rows["repair_mean_depth"])
case_rows["restart_depth_improvement_pct"] = _pct(
    case_rows["original_mean_depth"], case_rows["restart_mean_depth"])

fresh30_case_comparison = case_rows
print(f"fresh30_case_comparison: {len(fresh30_case_comparison)} cases.")
display(fresh30_case_comparison.round(3))


fresh30_case_comparison: 24 cases.


,case_id,circuit,original_mean_swaps,repair_mean_swaps,restart_mean_swaps,original_mean_depth,repair_mean_depth,restart_mean_depth,repair_swap_improvement,restart_swap_improvement,repair_swap_improvement_pct,restart_swap_improvement_pct,repair_depth_improvement,restart_depth_improvement,repair_depth_improvement_pct,restart_depth_improvement_pct
0,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,21.067,22.033,23.433,80.467,77.933,79.000,-0.967,-2.367,-4.589,-11.234,2.533,1.467,3.148,1.823
1,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,22.267,21.300,23.433,82.633,80.533,79.000,0.967,-1.167,4.341,-5.240,2.100,3.633,2.541,4.397
2,mini_alu_305.qasm::P1,mini_alu_305.qasm,42.767,43.900,41.967,158.800,158.767,152.567,-1.133,0.800,-2.650,1.871,0.033,6.233,0.021,3.925
3,mini_alu_305.qasm::P11,mini_alu_305.qasm,44.767,44.400,45.100,158.700,157.367,154.900,0.367,-0.333,0.819,-0.745,1.333,3.800,0.840,2.394
4,mini_alu_305.qasm::P12,mini_alu_305.qasm,41.700,42.800,42.933,154.733,156.733,161.367,-1.100,-1.233,-2.638,-2.958,-2.000,-6.633,-1.293,-4.287
5,mini_alu_305.qasm::P13,mini_alu_305.qasm,46.400,46.200,44.100,161.100,164.000,161.400,0.200,2.300,0.431,4.957,-2.900,-0.300,-1.800,-0.186
6,mini_alu_305.qasm::P14,mini_alu_305.qasm,44.800,45.000,44.133,159.000,159.567,159.067,-0.200,0.667,-0.446,1.488,-0.567,-0.067,-0.356,-0.042
7,mini_alu_305.qasm::P2,mini_alu_305.qasm,41.967,43.033,44.167,152.600,154.033,154.700,-1.067,-2.200,-2.542,-5.242,-1.433,-2.100,-0.939,-1.376
8,mini_alu_305.qasm::P3,mini_alu_305.qasm,44.367,45.500,42.033,157.733,157.567,158.133,-1.133,2.333,-2.554,5.259,0.167,-0.400,0.106,-0.254
9,mini_alu_305.qasm::P4,mini_alu_305.qasm,44.133,43.933,45.300,151.133,149.800,160.433,0.200,-1.167,0.453,-2.644,1.333,-9.300,0.882,-6.154


## Step 9 — Repair vs Restart-1 direct comparison

Per case: WIN = Repair fresh mean strictly lower; LOSS = strictly higher; TIE = equal.
Paired difference uses `restart_mean - repair_mean` (positive = Repair better).


In [11]:
# ------------------------------------------------------------
# STEP 9 — direct Repair vs Restart-1 (fresh means)
# ------------------------------------------------------------
cmp_df = fresh30_case_comparison.copy()
cmp_df["paired_swap_diff"] = cmp_df["restart_mean_swaps"] - cmp_df["repair_mean_swaps"]
cmp_df["paired_depth_diff"] = cmp_df["restart_mean_depth"] - cmp_df["repair_mean_depth"]

def _wlt(s):
    return np.where(np.asarray(s) > 0, "WIN", np.where(np.asarray(s) < 0, "LOSS", "TIE"))

cmp_df["swap_WLT"] = _wlt(cmp_df["paired_swap_diff"])
cmp_df["depth_WLT"] = _wlt(cmp_df["paired_depth_diff"])
fresh30_repair_vs_restart = cmp_df[[
    "case_id", "circuit", "repair_mean_swaps", "restart_mean_swaps",
    "paired_swap_diff", "swap_WLT", "repair_mean_depth", "restart_mean_depth",
    "paired_depth_diff", "depth_WLT"]].copy()

def _summ(col):
    v = fresh30_repair_vs_restart[col].value_counts()
    return (int(v.get("WIN", 0)), int(v.get("LOSS", 0)), int(v.get("TIE", 0)))

swap_wlt = _summ("swap_WLT")
depth_wlt = _summ("depth_WLT")
print(f"Repair vs Restart-1 SWAP  W/L/T = {swap_wlt[0]} / {swap_wlt[1]} / {swap_wlt[2]}")
print(f"Repair vs Restart-1 Depth W/L/T = {depth_wlt[0]} / {depth_wlt[1]} / {depth_wlt[2]}")
print(f"paired SWAP diff (restart-repair): mean={cmp_df['paired_swap_diff'].mean():.3f}, "
      f"median={cmp_df['paired_swap_diff'].median():.3f}")
print(f"paired Depth diff (restart-repair): mean={cmp_df['paired_depth_diff'].mean():.3f}, "
      f"median={cmp_df['paired_depth_diff'].median():.3f}")
display(fresh30_repair_vs_restart.round(3))


Repair vs Restart-1 SWAP  W/L/T = 11 / 13 / 0
Repair vs Restart-1 Depth W/L/T = 15 / 9 / 0
paired SWAP diff (restart-repair): mean=-0.815, median=-1.117
paired Depth diff (restart-repair): mean=1.826, median=0.617


,case_id,circuit,repair_mean_swaps,restart_mean_swaps,paired_swap_diff,swap_WLT,repair_mean_depth,restart_mean_depth,paired_depth_diff,depth_WLT
0,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,22.033,23.433,1.400,WIN,77.933,79.000,1.067,WIN
1,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,21.300,23.433,2.133,WIN,80.533,79.000,-1.533,LOSS
2,mini_alu_305.qasm::P1,mini_alu_305.qasm,43.900,41.967,-1.933,LOSS,158.767,152.567,-6.200,LOSS
3,mini_alu_305.qasm::P11,mini_alu_305.qasm,44.400,45.100,0.700,WIN,157.367,154.900,-2.467,LOSS
4,mini_alu_305.qasm::P12,mini_alu_305.qasm,42.800,42.933,0.133,WIN,156.733,161.367,4.633,WIN
5,mini_alu_305.qasm::P13,mini_alu_305.qasm,46.200,44.100,-2.100,LOSS,164.000,161.400,-2.600,LOSS
6,mini_alu_305.qasm::P14,mini_alu_305.qasm,45.000,44.133,-0.867,LOSS,159.567,159.067,-0.500,LOSS
7,mini_alu_305.qasm::P2,mini_alu_305.qasm,43.033,44.167,1.133,WIN,154.033,154.700,0.667,WIN
8,mini_alu_305.qasm::P3,mini_alu_305.qasm,45.500,42.033,-3.467,LOSS,157.567,158.133,0.567,WIN
9,mini_alu_305.qasm::P4,mini_alu_305.qasm,43.933,45.300,1.367,WIN,149.800,160.433,10.633,WIN


## Step 10 — Success rates (improved / worsened / tied)

Reported for Repair-vs-Original and Restart-1-vs-Original (SWAP and depth),
Repair-vs-Restart-1, overall and by circuit where counts permit.


In [12]:
# ------------------------------------------------------------
# STEP 10 — success rates
# ------------------------------------------------------------
def _iwt(s):
    s = np.asarray(s, dtype=float)
    return (int((s > 0).sum()), int((s < 0).sum()), int((s == 0).sum()))

rate_rows = []
for label, col in [
    ("Repair vs Original SWAP", "repair_swap_improvement"),
    ("Repair vs Original Depth", "repair_depth_improvement"),
    ("Restart-1 vs Original SWAP", "restart_swap_improvement"),
    ("Restart-1 vs Original Depth", "restart_depth_improvement"),
    ("Repair vs Restart-1 SWAP", "paired_swap_diff"),
    ("Repair vs Restart-1 Depth", "paired_depth_diff"),
]:
    base = fresh30_case_comparison if "paired_" not in col else cmp_df
    i, w, t = _iwt(base[col])
    rate_rows.append({"comparison": label, "improved_WIN": i, "worsened_LOSS": w, "tied": t,
                      "n": len(base)})
success_rates = pd.DataFrame(rate_rows)
print("Overall success rates (positive = better for the first-named alternative):")
display(success_rates)

print("\nBy circuit:")
by_circuit_rows = []
for circuit, g in fresh30_case_comparison.groupby("circuit"):
    idx = cmp_df["circuit"] == circuit
    by_circuit_rows.append({
        "circuit": circuit, "n": len(g),
        "repair_swap_WLT": _iwt(g["repair_swap_improvement"]),
        "repair_depth_WLT": _iwt(g["repair_depth_improvement"]),
        "restart_swap_WLT": _iwt(g["restart_swap_improvement"]),
        "restart_depth_WLT": _iwt(g["restart_depth_improvement"]),
        "repair_vs_restart_swap_WLT": _iwt(cmp_df[idx]["paired_swap_diff"]),
        "repair_vs_restart_depth_WLT": _iwt(cmp_df[idx]["paired_depth_diff"]),
    })
display(pd.DataFrame(by_circuit_rows))


Overall success rates (positive = better for the first-named alternative):


,comparison,improved_WIN,worsened_LOSS,tied,n
0,Repair vs Original SWAP,16,7,1,24
1,Repair vs Original Depth,15,9,0,24
2,Restart-1 vs Original SWAP,15,9,0,24
3,Restart-1 vs Original Depth,14,10,0,24
4,Repair vs Restart-1 SWAP,11,13,0,24
5,Repair vs Restart-1 Depth,15,9,0,24



By circuit:


,circuit,n,repair_swap_WLT,repair_depth_WLT,restart_swap_WLT,restart_depth_WLT,repair_vs_restart_swap_WLT,repair_vs_restart_depth_WLT
0,alu-bdd_288.qasm,2,"(1, 1, 0)","(2, 0, 0)","(0, 2, 0)","(2, 0, 0)","(2, 0, 0)","(1, 1, 0)"
1,mini_alu_305.qasm,12,"(7, 5, 0)","(7, 5, 0)","(8, 4, 0)","(5, 7, 0)","(5, 7, 0)","(7, 5, 0)"
2,rd73_140.qasm,1,"(1, 0, 0)","(1, 0, 0)","(1, 0, 0)","(1, 0, 0)","(1, 0, 0)","(1, 0, 0)"
3,sym9_146.qasm,5,"(3, 1, 1)","(1, 4, 0)","(4, 1, 0)","(3, 2, 0)","(1, 4, 0)","(2, 3, 0)"
4,system9.qasm,4,"(4, 0, 0)","(4, 0, 0)","(2, 2, 0)","(3, 1, 0)","(2, 2, 0)","(4, 0, 0)"


## Step 11 — Old (5-seed) vs fresh (30-seed) B=1 comparison

Old means come from the frozen Experiment-3 artifacts only (never re-routed):
- Original: `experiment1_raw.csv` means over seeds 42–46.
- Repair: `experiment3_budget_raw_routing.csv`, `strategy == "repair"`, `candidate_index == 1`.
- Restart-1: same file, `strategy == "restart"`, `candidate_index == 1`.

Joined to fresh means on stable `(case_id, alternative)` identity — never by row position.
Sign agreement asks whether old and fresh agree that an alternative improved / worsened
relative to Original (ties counted separately). Correlations are descriptive only (n=24).


In [13]:
# ------------------------------------------------------------
# STEP 11 — old-vs-fresh stability (old data read from frozen artifacts)
# ------------------------------------------------------------
old_orig = (exp1_raw[exp1_raw["routing_seed"].isin(OLD_ROUTING_SEEDS)]
            .groupby(["circuit", "plan_id"])
            .agg(old_original_mean_swaps=("swap_count", "mean"),
                 old_original_mean_depth=("routed_depth", "mean"),
                 old_n_seeds=("routing_seed", "size"))
            .reset_index().rename(columns={"plan_id": "seed_plan_id"}))
# exp1 covers all 50 plans; restrict to our 24 frozen cases and require 5 old seeds each.
old_orig = old_orig.merge(b1_frozen_ledger[["circuit", "starting_plan_id"]],
                           left_on=["circuit", "seed_plan_id"],
                           right_on=["circuit", "starting_plan_id"], how="inner")
assert len(old_orig) == N and (old_orig["old_n_seeds"] == 5).all()
old_rep = (old_repair_b1.groupby(["circuit", "seed_plan_id"])
           .agg(old_repair_mean_swaps=("swaps_after", "mean"),
                old_repair_mean_depth=("routed_depth_after", "mean"))
           .reset_index())
old_rst = (old_restart_b1.groupby(["circuit", "seed_plan_id"])
           .agg(old_restart_mean_swaps=("swaps_after", "mean"),
                old_restart_mean_depth=("routed_depth_after", "mean"))
           .reset_index())
assert len(old_rep) == N and len(old_rst) == N

# --- build the matched table directly per case on stable (case_id, alternative) identity ---
rows = []
for _, c in fresh30_case_comparison.iterrows():
    circ, seed = c["circuit"], c["case_id"].split("::")[1]
    oo = old_orig[(old_orig["circuit"] == circ) & (old_orig["seed_plan_id"] == seed)].iloc[0]
    orr = old_rep[(old_rep["circuit"] == circ) & (old_rep["seed_plan_id"] == seed)].iloc[0]
    ors = old_rst[(old_rst["circuit"] == circ) & (old_rst["seed_plan_id"] == seed)].iloc[0]
    rows.append({
        "case_id": c["case_id"], "circuit": circ,
        "old_original_mean_swaps": float(oo["old_original_mean_swaps"]),
        "old_original_mean_depth": float(oo["old_original_mean_depth"]),
        "old_repair_mean_swaps": float(orr["old_repair_mean_swaps"]),
        "old_repair_mean_depth": float(orr["old_repair_mean_depth"]),
        "old_restart_mean_swaps": float(ors["old_restart_mean_swaps"]),
        "old_restart_mean_depth": float(ors["old_restart_mean_depth"]),
        "fresh_original_mean_swaps": float(c["original_mean_swaps"]),
        "fresh_original_mean_depth": float(c["original_mean_depth"]),
        "fresh_repair_mean_swaps": float(c["repair_mean_swaps"]),
        "fresh_repair_mean_depth": float(c["repair_mean_depth"]),
        "fresh_restart_mean_swaps": float(c["restart_mean_swaps"]),
        "fresh_restart_mean_depth": float(c["restart_mean_depth"]),
    })
old_vs_fresh_base = pd.DataFrame(rows)
for prefix in ["old", "fresh"]:
    old_vs_fresh_base[f"{prefix}_repair_swap_imp"] = (
        old_vs_fresh_base[f"{prefix}_original_mean_swaps"] - old_vs_fresh_base[f"{prefix}_repair_mean_swaps"])
    old_vs_fresh_base[f"{prefix}_restart_swap_imp"] = (
        old_vs_fresh_base[f"{prefix}_original_mean_swaps"] - old_vs_fresh_base[f"{prefix}_restart_mean_swaps"])
    old_vs_fresh_base[f"{prefix}_repair_depth_imp"] = (
        old_vs_fresh_base[f"{prefix}_original_mean_depth"] - old_vs_fresh_base[f"{prefix}_repair_mean_depth"])
    old_vs_fresh_base[f"{prefix}_restart_depth_imp"] = (
        old_vs_fresh_base[f"{prefix}_original_mean_depth"] - old_vs_fresh_base[f"{prefix}_restart_mean_depth"])
    old_vs_fresh_base[f"{prefix}_paired_swap"] = (
        old_vs_fresh_base[f"{prefix}_restart_mean_swaps"] - old_vs_fresh_base[f"{prefix}_repair_mean_swaps"])
    old_vs_fresh_base[f"{prefix}_paired_depth"] = (
        old_vs_fresh_base[f"{prefix}_restart_mean_depth"] - old_vs_fresh_base[f"{prefix}_repair_mean_depth"])

def _sign_agreement(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    sa, sb = np.sign(a), np.sign(b)
    agree = int(((sa == sb) & (sa != 0)).sum())
    disagree = int(((sa != sb) & (sa != 0) & (sb != 0)).sum())
    tied = int(((sa == 0) | (sb == 0)).sum())
    return agree, disagree, tied

old_vs_fresh_comparison = old_vs_fresh_base  # matched cleanly on (case_id, alternative identity)
print("Old-vs-fresh matched cases:", len(old_vs_fresh_comparison), "(expect 24)")
for label, ocol, fcol in [
    ("Repair vs Original SWAP", "old_repair_swap_imp", "fresh_repair_swap_imp"),
    ("Repair vs Original Depth", "old_repair_depth_imp", "fresh_repair_depth_imp"),
    ("Restart vs Original SWAP", "old_restart_swap_imp", "fresh_restart_swap_imp"),
    ("Restart vs Original Depth", "old_restart_depth_imp", "fresh_restart_depth_imp"),
    ("Repair vs Restart SWAP", "old_paired_swap", "fresh_paired_swap"),
    ("Repair vs Restart Depth", "old_paired_depth", "fresh_paired_depth"),
]:
    a, d, t = _sign_agreement(old_vs_fresh_comparison[ocol], old_vs_fresh_comparison[fcol])
    r = float(np.corrcoef(old_vs_fresh_comparison[ocol], old_vs_fresh_comparison[fcol])[0, 1])
    print(f"{label}: sign agree={a} disagree={d} tied={t} | old-vs-fresh Pearson r={r:.3f} (n=24, descriptive)")
display(old_vs_fresh_comparison.round(3))


Old-vs-fresh matched cases: 24 (expect 24)
Repair vs Original SWAP: sign agree=19 disagree=3 tied=2 | old-vs-fresh Pearson r=0.849 (n=24, descriptive)
Repair vs Original Depth: sign agree=18 disagree=6 tied=0 | old-vs-fresh Pearson r=0.897 (n=24, descriptive)
Restart vs Original SWAP: sign agree=22 disagree=2 tied=0 | old-vs-fresh Pearson r=0.920 (n=24, descriptive)
Restart vs Original Depth: sign agree=20 disagree=4 tied=0 | old-vs-fresh Pearson r=0.849 (n=24, descriptive)
Repair vs Restart SWAP: sign agree=19 disagree=4 tied=1 | old-vs-fresh Pearson r=0.859 (n=24, descriptive)
Repair vs Restart Depth: sign agree=22 disagree=2 tied=0 | old-vs-fresh Pearson r=0.925 (n=24, descriptive)


,case_id,circuit,old_original_mean_swaps,old_original_mean_depth,old_repair_mean_swaps,old_repair_mean_depth,old_restart_mean_swaps,old_restart_mean_depth,fresh_original_mean_swaps,fresh_original_mean_depth,fresh_repair_mean_swaps,fresh_repair_mean_depth,fresh_restart_mean_swaps,fresh_restart_mean_depth,old_repair_swap_imp,old_restart_swap_imp,old_repair_depth_imp,old_restart_depth_imp,old_paired_swap,old_paired_depth,fresh_repair_swap_imp,fresh_restart_swap_imp,fresh_repair_depth_imp,fresh_restart_depth_imp,fresh_paired_swap,fresh_paired_depth
0,alu-bdd_288.qasm::P0,alu-bdd_288.qasm,21.2,80.2,21.6,77.8,23.2,79.6,21.067,80.467,22.033,77.933,23.433,79.000,-0.4,-2.0,2.4,0.6,1.6,1.8,-0.967,-2.367,2.533,1.467,1.400,1.067
1,alu-bdd_288.qasm::P5,alu-bdd_288.qasm,22.0,82.6,21.4,79.8,23.2,79.6,22.267,82.633,21.300,80.533,23.433,79.000,0.6,-1.2,2.8,3.0,1.8,-0.2,0.967,-1.167,2.100,3.633,2.133,-1.533
2,mini_alu_305.qasm::P1,mini_alu_305.qasm,43.0,158.2,43.0,158.6,42.6,153.6,42.767,158.800,43.900,158.767,41.967,152.567,0.0,0.4,-0.4,4.6,-0.4,-5.0,-1.133,0.800,0.033,6.233,-1.933,-6.200
3,mini_alu_305.qasm::P11,mini_alu_305.qasm,44.2,157.4,45.2,158.6,45.0,155.2,44.767,158.700,44.400,157.367,45.100,154.900,-1.0,-0.8,-1.2,2.2,-0.2,-3.4,0.367,-0.333,1.333,3.800,0.700,-2.467
4,mini_alu_305.qasm::P12,mini_alu_305.qasm,41.8,154.8,43.0,157.2,42.2,161.2,41.700,154.733,42.800,156.733,42.933,161.367,-1.2,-0.4,-2.4,-6.4,-0.8,4.0,-1.100,-1.233,-2.000,-6.633,0.133,4.633
5,mini_alu_305.qasm::P13,mini_alu_305.qasm,45.4,159.6,47.0,165.2,46.4,164.0,46.400,161.100,46.200,164.000,44.100,161.400,-1.6,-1.0,-5.6,-4.4,-0.6,-1.2,0.200,2.300,-2.900,-0.300,-2.100,-2.600
6,mini_alu_305.qasm::P14,mini_alu_305.qasm,45.2,159.4,45.6,162.2,44.0,159.2,44.800,159.000,45.000,159.567,44.133,159.067,-0.4,1.2,-2.8,0.2,-1.6,-3.0,-0.200,0.667,-0.567,-0.067,-0.867,-0.500
7,mini_alu_305.qasm::P2,mini_alu_305.qasm,41.6,156.0,43.6,152.6,44.8,154.6,41.967,152.600,43.033,154.033,44.167,154.700,-2.0,-3.2,3.4,1.4,1.2,2.0,-1.067,-2.200,-1.433,-2.100,1.133,0.667
8,mini_alu_305.qasm::P3,mini_alu_305.qasm,44.4,157.4,45.4,157.2,42.0,158.4,44.367,157.733,45.500,157.567,42.033,158.133,-1.0,2.4,0.2,-1.0,-3.4,1.2,-1.133,2.333,0.167,-0.400,-3.467,0.567
9,mini_alu_305.qasm::P4,mini_alu_305.qasm,44.2,156.0,44.0,152.6,44.4,159.8,44.133,151.133,43.933,149.800,45.300,160.433,0.2,-0.2,3.4,-3.8,0.4,7.2,0.200,-1.167,1.333,-9.300,1.367,10.633


## Step 12 — Core presentation table (fresh 30-seed B=1 evaluation)

Descriptive summary only: mean / median improvement (positive = better), counts of
improved / worsened / tied cases. Negative results are shown, not hidden.


In [14]:
# ------------------------------------------------------------
# STEP 12 — core presentation table + final-summary frame
# ------------------------------------------------------------
def _row(label, vals):
    vals = np.asarray(vals, dtype=float)
    i, w, t = int((vals > 0).sum()), int((vals < 0).sum()), int((vals == 0).sum())
    return {"comparison": label, "mean_improvement": float(vals.mean()),
            "median_improvement": float(np.median(vals)),
            "improved": i, "worsened": w, "tied": t, "n": len(vals)}

core_rows = [
    _row("Repair vs Original SWAP", fresh30_case_comparison["repair_swap_improvement"]),
    _row("Repair vs Original Depth", fresh30_case_comparison["repair_depth_improvement"]),
    _row("Restart-1 vs Original SWAP", fresh30_case_comparison["restart_swap_improvement"]),
    _row("Restart-1 vs Original Depth", fresh30_case_comparison["restart_depth_improvement"]),
    _row("Repair vs Restart-1 SWAP (restart - repair)",
         fresh30_repair_vs_restart["paired_swap_diff"]),
    _row("Repair vs Restart-1 Depth (restart - repair)",
         fresh30_repair_vs_restart["paired_depth_diff"]),
]
core_table = pd.DataFrame(core_rows)
print("Fresh 30-seed B=1 evaluation (positive = first-named or Repair better):")
display(core_table.round(3))

fresh30_final_summary = core_table.copy()
print("\nSaved as experiment3_b1_fresh30_final_summary.csv in STEP 14.")


Fresh 30-seed B=1 evaluation (positive = first-named or Repair better):


,comparison,mean_improvement,median_improvement,improved,worsened,tied,n
0,Repair vs Original SWAP,0.669,0.383,16,7,1,24
1,Repair vs Original Depth,2.590,1.083,15,9,0,24
2,Restart-1 vs Original SWAP,1.485,1.667,15,9,0,24
3,Restart-1 vs Original Depth,0.764,0.667,14,10,0,24
4,Repair vs Restart-1 SWAP (restart - repair),-0.815,-1.117,11,13,0,24
5,Repair vs Restart-1 Depth (restart - repair),1.826,0.617,15,9,0,24



Saved as experiment3_b1_fresh30_final_summary.csv in STEP 14.


## Step 13 — Plots (matplotlib only, 300 dpi)

Horizontal zero line: positive = improvement (Repair better for paired differences).


In [15]:
# ------------------------------------------------------------
# STEP 13 — presentation plots (matplotlib only)
# ------------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt

short = fresh30_case_comparison["circuit"].str.replace(".qasm", "", regex=False) \
    + "|" + fresh30_case_comparison["case_id"].str.split("::").str[1]

def barh_imp(values, labels, title, xlabel, fname, color):
    order = np.argsort(np.asarray(values, dtype=float))
    v = np.asarray(values, dtype=float)[order]
    lab = np.asarray(labels)[order]
    fig, ax = plt.subplots(figsize=(10, 8))
    y = np.arange(len(v))
    ax.barh(y, v, color=color)
    ax.set_yticks(y)
    ax.set_yticklabels(lab, fontsize=7)
    ax.axvline(0.0, color="black", linewidth=1.0)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=300)
    plt.close(fig)
    print("saved", fname)

barh_imp(fresh30_case_comparison["repair_swap_improvement"], short,
         "Repair vs Original: case-level SWAP improvement (fresh 30-seed mean, + = better)",
         "Original mean SWAP - Repair mean SWAP", "fresh30_repair_vs_original_swap.png", "steelblue")
barh_imp(fresh30_case_comparison["restart_swap_improvement"], short,
         "Restart-1 vs Original: case-level SWAP improvement (fresh 30-seed mean, + = better)",
         "Original mean SWAP - Restart mean SWAP", "fresh30_restart_vs_original_swap.png", "darkorange")
barh_imp(fresh30_case_comparison["repair_depth_improvement"], short,
         "Repair vs Original: case-level routed-depth improvement (fresh 30-seed mean, + = better)",
         "Original mean depth - Repair mean depth", "fresh30_repair_vs_original_depth.png", "steelblue")
barh_imp(fresh30_case_comparison["restart_depth_improvement"], short,
         "Restart-1 vs Original: case-level routed-depth improvement (fresh 30-seed mean, + = better)",
         "Original mean depth - Restart mean depth", "fresh30_restart_vs_original_depth.png", "darkorange")
barh_imp(fresh30_repair_vs_restart["paired_swap_diff"], short,
         "Repair vs Restart-1: paired SWAP difference by case (restart - repair, + = Repair better)",
         "Restart mean SWAP - Repair mean SWAP", "fresh30_repair_vs_restart_swap.png", "seagreen")
barh_imp(fresh30_repair_vs_restart["paired_depth_diff"], short,
         "Repair vs Restart-1: paired routed-depth difference by case (restart - repair, + = Repair better)",
         "Restart mean depth - Repair mean depth", "fresh30_repair_vs_restart_depth.png", "seagreen")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, ocol, fcol, title in [
    (axes[0], "old_repair_swap_imp", "fresh_repair_swap_imp", "Repair vs Original SWAP"),
    (axes[1], "old_restart_swap_imp", "fresh_restart_swap_imp", "Restart vs Original SWAP"),
]:
    ax.scatter(old_vs_fresh_comparison[ocol], old_vs_fresh_comparison[fcol], s=28)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, color="grey", linewidth=1.0)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.axvline(0.0, color="black", linewidth=0.8)
    ax.set_title(title + ": old 5-seed vs fresh 30-seed improvement")
    ax.set_xlabel("old improvement (+ = better)")
    ax.set_ylabel("fresh improvement (+ = better)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fresh30_old_vs_fresh_swap.png", dpi=300)
plt.close(fig)
print("saved fresh30_old_vs_fresh_swap.png")
print("\nAll figures saved to:", FIG_DIR)


saved fresh30_repair_vs_original_swap.png
saved fresh30_restart_vs_original_swap.png
saved fresh30_repair_vs_original_depth.png
saved fresh30_restart_vs_original_depth.png
saved fresh30_repair_vs_restart_swap.png
saved fresh30_repair_vs_restart_depth.png
saved fresh30_old_vs_fresh_swap.png

All figures saved to: C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment3_b1_fresh30_validation\figures


## Step 14 — Save new results only

Everything below lands in `experiment3_b1_fresh30_validation/` (+ `figures/`).
Nothing under `experiment3_results/` is touched.


In [16]:
# ------------------------------------------------------------
# STEP 14 — save ONLY new validation results
# ------------------------------------------------------------
fresh30_raw.to_csv(OUT_DIR / "experiment3_b1_fresh30_raw.csv", index=False)
fresh30_alternative_summary.to_csv(
    OUT_DIR / "experiment3_b1_fresh30_alternative_summary.csv", index=False)
fresh30_case_comparison.to_csv(
    OUT_DIR / "experiment3_b1_fresh30_case_comparison.csv", index=False)
fresh30_repair_vs_restart.to_csv(
    OUT_DIR / "experiment3_b1_fresh30_repair_vs_restart.csv", index=False)
old_vs_fresh_comparison.to_csv(
    OUT_DIR / "experiment3_b1_old_vs_fresh_comparison.csv", index=False)
fresh30_final_summary.to_csv(
    OUT_DIR / "experiment3_b1_fresh30_final_summary.csv", index=False)

print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {p.name}: {pd.read_csv(p).shape}")
print("Figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  figures/{p.name}")


Saved to C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment3_b1_fresh30_validation
  experiment3_b1_fresh30_alternative_summary.csv: (72, 16)
  experiment3_b1_fresh30_case_comparison.csv: (24, 16)
  experiment3_b1_fresh30_final_summary.csv: (6, 7)
  experiment3_b1_fresh30_partial_checkpoint.csv: (2160, 15)
  experiment3_b1_fresh30_raw.csv: (2160, 15)
  experiment3_b1_fresh30_repair_vs_restart.csv: (24, 10)
  experiment3_b1_old_vs_fresh_comparison.csv: (24, 26)
Figures:
  figures/fresh30_old_vs_fresh_swap.png
  figures/fresh30_repair_vs_original_depth.png
  figures/fresh30_repair_vs_original_swap.png
  figures/fresh30_repair_vs_restart_depth.png
  figures/fresh30_repair_vs_restart_swap.png
  figures/fresh30_restart_vs_original_depth.png
  figures/fresh30_restart_vs_original_swap.png


## Step 15 — Final summary (descriptive only)

This validation answers: *does the previously selected B=1 targeted repair retain its
apparent advantage under 30 new independent routing seeds, and does it clearly outperform
one equal-budget restart?* Do **not** claim success if Repair only improves on Original
but does not clearly outperform Restart-1. Mixed results must be reported as mixed.


In [17]:
# ------------------------------------------------------------
# STEP 15 — final printed summary
# ------------------------------------------------------------
print("=" * 70)
print("EXPERIMENT 3-B1 FRESH 30-SEED VALIDATION — FINAL SUMMARY")
print("=" * 70)
print(f"1. Frozen B=1 cases: {len(b1_frozen_ledger)}")
print(f"2. Reconstructed: Original {int(verification_df[verification_df['alternative']=='Original']['ok'].sum())}"
      f"/{len(b1_frozen_ledger)}; Repair "
      f"{int(verification_df[verification_df['alternative']=='Repair']['ok'].sum())}/{len(b1_frozen_ledger)}; "
      f"Restart-1 {int(verification_df[verification_df['alternative']=='Restart-1']['ok'].sum())}"
      f"/{len(b1_frozen_ledger)}")
print(f"3. Fresh physical routing runs: {len(fresh30_raw)} "
      f"({len(b1_frozen_ledger)} cases x 3 alternatives x 30 seeds 100-129)")
i, w, t = (int((fresh30_case_comparison['repair_swap_improvement'] > 0).sum()),
           int((fresh30_case_comparison['repair_swap_improvement'] < 0).sum()),
           int((fresh30_case_comparison['repair_swap_improvement'] == 0).sum()))
print(f"4a. Repair vs Original SWAP: improve/worse/tie = {i}/{w}/{t}")
i, w, t = (int((fresh30_case_comparison['repair_depth_improvement'] > 0).sum()),
           int((fresh30_case_comparison['repair_depth_improvement'] < 0).sum()),
           int((fresh30_case_comparison['repair_depth_improvement'] == 0).sum()))
print(f"4b. Repair vs Original Depth: improve/worse/tie = {i}/{w}/{t}")
i, w, t = (int((fresh30_case_comparison['restart_swap_improvement'] > 0).sum()),
           int((fresh30_case_comparison['restart_swap_improvement'] < 0).sum()),
           int((fresh30_case_comparison['restart_swap_improvement'] == 0).sum()))
print(f"5a. Restart-1 vs Original SWAP: improve/worse/tie = {i}/{w}/{t}")
i, w, t = (int((fresh30_case_comparison['restart_depth_improvement'] > 0).sum()),
           int((fresh30_case_comparison['restart_depth_improvement'] < 0).sum()),
           int((fresh30_case_comparison['restart_depth_improvement'] == 0).sum()))
print(f"5b. Restart-1 vs Original Depth: improve/worse/tie = {i}/{w}/{t}")
sw = fresh30_repair_vs_restart['swap_WLT'].value_counts()
dp = fresh30_repair_vs_restart['depth_WLT'].value_counts()
print(f"6a. Repair vs Restart-1 SWAP W/L/T = {int(sw.get('WIN', 0))}/{int(sw.get('LOSS', 0))}/{int(sw.get('TIE', 0))}")
print(f"6b. Repair vs Restart-1 Depth W/L/T = {int(dp.get('WIN', 0))}/{int(dp.get('LOSS', 0))}/{int(dp.get('TIE', 0))}")
print("7. Mean/median fresh improvements (positive = Repair/first-named better):")
print(core_table.round(3).to_string(index=False))
print("8. Old-vs-fresh sign agreement: see Step 11 output table above.")
print("=" * 70)


EXPERIMENT 3-B1 FRESH 30-SEED VALIDATION — FINAL SUMMARY
1. Frozen B=1 cases: 24
2. Reconstructed: Original 24/24; Repair 24/24; Restart-1 24/24
3. Fresh physical routing runs: 2160 (24 cases x 3 alternatives x 30 seeds 100-129)
4a. Repair vs Original SWAP: improve/worse/tie = 16/7/1
4b. Repair vs Original Depth: improve/worse/tie = 15/9/0
5a. Restart-1 vs Original SWAP: improve/worse/tie = 15/9/0
5b. Restart-1 vs Original Depth: improve/worse/tie = 14/10/0
6a. Repair vs Restart-1 SWAP W/L/T = 11/13/0
6b. Repair vs Restart-1 Depth W/L/T = 15/9/0
7. Mean/median fresh improvements (positive = Repair/first-named better):
                                  comparison  mean_improvement  median_improvement  improved  worsened  tied  n
                     Repair vs Original SWAP             0.669               0.383        16         7     1 24
                    Repair vs Original Depth             2.590               1.083        15         9     0 24
                  Restart-1 vs Origina